In [5]:
!python -m pip install transformers
!python -m pip install torch
!python -m pip install gradio
!python -m pip install pillow

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
import gradio as gr
from transformers import BlipProcessor, BlipForConditionalGeneration
#from transformers import pipeline
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Завантажуємо пайплайн для генерації підписів до зображень
#captioner = pipeline("image-text-to-text", model="Salesforce/blip-image-captioning-base")

def generate_caption(image):
    # Конвертуємо в RGB, якщо зображення в іншому форматі (наприклад, RGBA)
    if image.mode != "RGB":
        image = image.convert("RGB")
        
    #directly using PIL image there
    inputs = processor(images=image, return_tensors="pt")
    outputs = model.generate(
        **inputs, 
        max_new_tokens=30,          # максимальна довжина
        num_beams=5,                 # променевий пошук
        repetition_penalty=1.2,      # штрафує за повторення токенів
        no_repeat_ngram_size=2,      # забороняє повтори 2-грам
        early_stopping=True          # зупиняється, коли ймовірність падає
    )
    
    caption = processor.decode(outputs[0], skip_special_tokens=True)
    return caption

def caption_image(image):
    
    #takes PIL image and returns a caption
    try:
        caption = generate_caption(image)

        # Пайплайн повертає список словників, беремо перший результат
        #result = captioner(image)
        return caption
        #return result[0]['generated_text']
    except Exception as e:
        return f"Error: {str(e)}"

interface = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Image Captioning with BLIP",
    description="Upload an image to generate a caption"
)

interface.launch(server_name="127.0.0.1", server_port=7070)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

* Running on local URL:  http://127.0.0.1:7070
* To create a public link, set `share=True` in `launch()`.
